# ProductIQ Phase 4 — Step 4.14: Semantic Retrieval Evaluation

Load the semantic benchmark, run evaluation through `SemanticRetriever` (BGE + pgvector), and inspect aggregate and per-query P@K / R@K / MRR.

In [ ]:
from pathlib import Path

from productiq.config.settings import get_settings
from productiq.database.engine import create_engine_from_settings
from productiq.retrieval.embedding_encoder import create_bge_small_en_v15_encoder
from productiq.retrieval.evaluation import (
    SemanticRetrievalEvaluator,
    build_per_query_analysis_rows,
    load_semantic_retrieval_benchmark,
)
from productiq.retrieval.semantic_retriever import create_semantic_retriever_from_engine

benchmark_path = Path("resources/evaluation/semantic_retrieval_benchmark_v1.json")
benchmark = load_semantic_retrieval_benchmark(benchmark_path)
print("Queries:", len(benchmark.queries))
print("Categories:", sorted({q.category for q in benchmark.queries}))

settings = get_settings()
engine = create_engine_from_settings(settings)
encoder = create_bge_small_en_v15_encoder(device="cpu", allow_cpu_fallback=True)
retriever = create_semantic_retriever_from_engine(engine, encoder)
evaluator = SemanticRetrievalEvaluator(retrieval_top_k=50, k_values=(1, 5, 10, 20, 50))
output = evaluator.evaluate(benchmark, retriever, engine=engine)
result = output.result

print("Aggregate:", result.aggregate.model_dump())
print()
for row in result.per_query:
    print(row.query_id, row.query_text, f"({row.category})")
    print("  P@10:", row.metrics.precision_at_k.get(10))
    print("  R@10:", row.metrics.recall_at_k.get(10))
    print("  RR:", row.metrics.reciprocal_rank)
    print("  first relevant rank:", row.metrics.first_relevant_rank)
    print()

analysis = build_per_query_analysis_rows(output)
print("Sample analysis row keys:", list(analysis[0].keys()) if analysis else [])
engine.dispose()